In [16]:
from pystac_client import Client
import geopandas as gpd
import json
import xarray as xr
import zarr

In [17]:
harz_boundaries = gpd.read_file("resources/Nationalpark_Harz_boundaries.geojson")
bbox = harz_boundaries.geometry.values[0].bounds

In [22]:
catalog = Client.open("https://stac.core.eopf.eodc.eu")
results = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=["2025-04-30", "2025-05-01"],
)
items = results.item_collection()[0]
items

<Item id=S2A_MSIL2A_20250430T102701_N0511_R108_T32UPC_20250430T190517>

In [37]:
ds = xr.open_datatree(
    items.assets["SR_10m"].href,
    engine="eopf-zarr",
    chunks={},
    decode_timedelta=True,
)
ds

<xarray.DataTree>
Group: /
    Dimensions:  (y: 10980, x: 10980)
    Coordinates:
      * y        (y) int64 88kB 5800015 5800005 5799995 ... 5690245 5690235 5690225
      * x        (x) int64 88kB 600005 600015 600025 600035 ... 709775 709785 709795
    Data variables:
        b02      (y, x) float64 964MB dask.array<chunksize=(1830, 1830), meta=np.ndarray>
        b03      (y, x) float64 964MB dask.array<chunksize=(1830, 1830), meta=np.ndarray>
        b04      (y, x) float64 964MB dask.array<chunksize=(1830, 1830), meta=np.ndarray>
        b08      (y, x) float64 964MB dask.array<chunksize=(1830, 1830), meta=np.ndarray>

In [34]:
import zarr

store = items.assets["product"].href
root = zarr.open(store, mode='r')  # if not consolidated  # if consolidated metadata exists


print(root.tree())  # prints all groups and arrays recursively


/


In [26]:
import xarray as xr

# Open the datatree
ds = xr.open_datatree(
    items.assets["SR_10m"].href,
    engine="zarr",
    chunks={},  # optional, or set a chunk size
    decode_timedelta=True,
    consolidated=False,
)

# Pick a leaf and load a dataset
leaf = ds["2023-09-01"]  # example date leaf
leaf_ds = leaf.load()  # loads all variables in memory

# Access a variable
b04 = leaf_ds["B04"].values  # NumPy array


KeyError: 'Could not find node at 2023-09-01'